# IP-008 — Results notebook

Statistical pass + eight plots over the `done` cohort, exported into
`presentation/public/images/plots/` for the OpenCamp 2026 deck.

- Five a-priori Mann-Whitney U tests, Bonferroni-corrected at α / 5 = 0.01.
  ESLint dropped from the family because the analyser silently failed
  for 100 % of the cohort (0 / 1,295 done repos).
- Effect size: rank-biserial correlation, signed so positive ⇒ profane
  cohort stochastically larger.
- All numbers regenerable: `MONGO_URI` env var → cohort pull → table → plots → JSON.


In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pymongo import MongoClient

sys.path.insert(0, str(Path.cwd()))
import _plot_helpers as ph

ph.style_fiit()

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017/profanity")
PLOT_DIR = Path("../presentation/public/images/plots/")
RESULTS_JSON = Path("../presentation/results.json")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# A-priori family — eslint OMITTED (100 % missing across the cohort).
A_PRIORI = [
    "lizard_avg_ccn",
    "lizard_ccn_p99",
    "ruff_issues_per_kloc",
    "jscpd_duplicate_rate",
    "comment_to_code_ratio",
]
ALPHA = 0.05
ALPHA_CORRECTED = ALPHA / len(A_PRIORI)
print(f"alpha_corrected = {ALPHA} / {len(A_PRIORI)} = {ALPHA_CORRECTED:.4f}")

alpha_corrected = 0.05 / 5 = 0.0100


## 1 · Pull the cohort

Single Mongo aggregation, projected into a pandas DataFrame. We keep
nulls in the frame and drop them per metric at test time so the
missingness summary survives.

In [2]:
client = MongoClient(MONGO_URI)
db = client.get_default_database()

PROJ = {
    "_id": 0,
    "full_name": 1,
    "cohort": 1,
    "primary_language": 1,
    "loc_total": "$code_analysis.loc_total",
    "files_scanned": "$code_analysis.files_scanned",
    "ruff_issues_per_kloc": "$code_analysis.ruff_issues_per_kloc",
    "ruff_bug_per_kloc": "$code_analysis.ruff_bug_issues_per_kloc",
    "eslint_issues_per_kloc": "$code_analysis.eslint_issues_per_kloc",
    "bandit_per_kloc": "$code_analysis.bandit_issues_per_kloc",
    "lizard_avg_ccn": "$code_analysis.lizard_avg_ccn",
    "lizard_max_ccn": "$code_analysis.lizard_max_ccn",
    "lizard_ccn_p90": "$code_analysis.lizard_ccn_p90",
    "lizard_ccn_p99": "$code_analysis.lizard_ccn_p99",
    "jscpd_duplicate_rate": "$code_analysis.jscpd_duplicate_rate",
    "comment_to_code_ratio": "$code_analysis.comment_to_code_ratio",
    "comment_profanity_hits": "$code_analysis.comment_profanity_hits",
    "id_profanity_hits": "$code_analysis.identifier_profanity_hits",
    "comment_emoji_hits": "$code_analysis.comment_emoji_hits",
    "id_emoji_hits": "$code_analysis.identifier_emoji_hits",
    "tech_debt_markers": "$code_analysis.tech_debt_markers",
}

df = pd.DataFrame(list(db.repos.aggregate([
    {"$match": {"cohort": {"$in": ["profane", "clean"]}, "status": "done"}},
    {"$project": PROJ},
])))
print(f"rows: {len(df)}")
print(df["cohort"].value_counts())

rows: 1295
cohort
clean      688
profane    607
Name: count, dtype: int64


## 2 · Cohort + missingness summary

Quick-read sanity check before any test runs. The missingness column
documents which metrics the family-wise correction can use.

In [3]:
summary = pd.DataFrame({
    "metric": A_PRIORI + ["eslint_issues_per_kloc"],
})
summary["n_clean"] = summary["metric"].apply(
    lambda m: int(df.loc[df.cohort == "clean", m].notna().sum())
)
summary["n_prof"] = summary["metric"].apply(
    lambda m: int(df.loc[df.cohort == "profane", m].notna().sum())
)
summary["pct_missing"] = summary["metric"].apply(
    lambda m: 100.0 * (1.0 - df[m].notna().mean())
)
summary

,metric,n_clean,n_prof,pct_missing
0,lizard_avg_ccn,570,508,16.756757
1,lizard_ccn_p99,565,502,17.606178
2,ruff_issues_per_kloc,112,96,83.938224
3,jscpd_duplicate_rate,653,567,5.791506
4,comment_to_code_ratio,684,604,0.540541
5,eslint_issues_per_kloc,0,0,100.000000


## 3 · Bonferroni — what we are correcting and why

We run **five** a-priori tests on the `done` cohort: `lizard_avg_ccn`,
`lizard_ccn_p99`, `ruff_issues_per_kloc`, `jscpd_duplicate_rate`,
`comment_to_code_ratio`.

If we used the raw α = 0.05 per test, the chance of seeing **at least one**
false positive across five independent tests would be

  $1 - (1 - 0.05)^5 = 1 - 0.95^5 \approx 22.6\,\%$.

Bonferroni keeps the family-wise error rate at 5 % by dividing α by the
number of tests:

  $\alpha_{corrected} = 0.05 / 5 = 0.01$.

A metric counts as significant only if its raw p-value sits below 0.01.
This is the conservative choice; Benjamini-Hochberg FDR would be more
forgiving but trades a different error guarantee. For a one-shot
conference talk, conservative is correct.

ESLint is **not in the family** — the analyser silent-failed for 100 % of
the cohort, so the test was never run. Including a never-run test in the
denominator would inflate the correction dishonestly.

## 4 · A-priori MWU table (pooled, all `done` repos)

In [4]:
results_pooled = ph.mwu_table(df, A_PRIORI, ALPHA_CORRECTED)
results_pooled = results_pooled.sort_values("p")
results_pooled

,metric,n_clean,n_prof,median_clean,median_prof,U,p,r_rb,significant
1,lizard_ccn_p99,565,502,10.000000,12.550000,121773.5,0.000066,0.141321,True
0,lizard_avg_ccn,570,508,2.132862,2.333236,125807.0,0.000201,0.131047,True
4,comment_to_code_ratio,684,604,0.018597,0.015827,211431.5,0.464886,-0.023544,False
2,ruff_issues_per_kloc,112,96,38.222499,48.487166,5068.5,0.478053,0.057199,False
3,jscpd_duplicate_rate,653,567,0.048642,0.047094,184318.0,0.894929,0.004362,False


## 5 · Python subset (n=112 vs 96)

In [5]:
df_py = df[df["primary_language"] == "python"]
PY_METRICS = ["ruff_issues_per_kloc", "ruff_bug_per_kloc", "bandit_per_kloc",
              "lizard_avg_ccn", "comment_profanity_hits", "id_profanity_hits"]
results_py = ph.mwu_table(df_py, PY_METRICS, ALPHA_CORRECTED)
results_py

,metric,n_clean,n_prof,median_clean,median_prof,U,p,r_rb,significant
0,ruff_issues_per_kloc,112,96,38.222499,48.487166,5068.5,4.780533e-01,0.057199,False
1,ruff_bug_per_kloc,112,96,8.184902,10.023757,5301.5,8.642105e-01,0.013858,False
2,bandit_per_kloc,112,96,1.026080,1.949075,4965.0,3.410965e-01,0.076451,False
3,lizard_avg_ccn,108,93,2.720298,3.102489,4468.0,1.782642e-01,0.110315,False
4,comment_profanity_hits,112,96,0.000000,0.000000,3458.5,2.999615e-08,0.356678,True
5,id_profanity_hits,112,96,0.000000,0.000000,4767.5,8.477720e-03,0.113188,True


## 6 · JS / TS subset — lizard only

ESLint per-kLOC is dropped here for the same 100 %-missing reason as
the pooled family. Lizard covers JS/TS via tree-sitter.

In [6]:
js_filter = df["primary_language"].isin(["javascript", "typescript", "tsx", "jsx"])
df_js = df[js_filter]
JS_METRICS = ["lizard_avg_ccn", "lizard_ccn_p99", "comment_profanity_hits",
              "id_profanity_hits", "comment_emoji_hits"]
results_js = ph.mwu_table(df_js, JS_METRICS, ALPHA_CORRECTED)
results_js

,metric,n_clean,n_prof,median_clean,median_prof,U,p,r_rb,significant
0,lizard_avg_ccn,176,125,1.665162,1.826087,10540.5,0.537313,0.041773,False
1,lizard_ccn_p99,175,125,7.820000,10.000000,9745.5,0.107642,0.108983,False
2,comment_profanity_hits,179,128,0.000000,0.000000,9234.0,0.000011,0.193959,True
3,id_profanity_hits,179,128,0.000000,0.000000,9377.0,0.000004,0.181477,True
4,comment_emoji_hits,179,128,0.000000,0.000000,11065.5,0.337553,0.034087,False


## 7 · Descriptive secondary panel

Comment + identifier profanity / emoji counts. Profanity rows are
significant **by construction** — cohort A is selected on commit-message
profanity, and the same authors leave profane comments. Reported here
for completeness; not subject to multiple-testing correction.

In [7]:
SECONDARY = ["comment_profanity_hits", "id_profanity_hits",
             "comment_emoji_hits", "id_emoji_hits", "tech_debt_markers"]
descriptive = ph.mwu_table(df, SECONDARY, ALPHA_CORRECTED)
descriptive

,metric,n_clean,n_prof,median_clean,median_prof,U,p,r_rb,significant
0,comment_profanity_hits,688,607,0.0,0.0,173438.5,2.910263e-12,0.169388,True
1,id_profanity_hits,688,607,0.0,0.0,183567.0,3.391295e-10,0.120881,True
2,comment_emoji_hits,688,607,0.0,0.0,202086.5,5.328944e-02,0.032190,False
3,id_emoji_hits,688,607,0.0,0.0,209415.0,1.842519e-01,-0.002907,False
4,tech_debt_markers,688,607,0.0,0.0,198150.0,7.120401e-02,0.051042,False


## 8 · Plots

Eight figures, written into `presentation/public/images/plots/`. Sized for
16:9 slide embedding (1920 × 1080), or 960 × 1080 each for the two-up
side-by-side block on Slide 50 (forest + ECDF).

### fig 01 — cohort funnel

Bar chart `done` / `failed` / `missing` per cohort. Visual proof of
completion.

In [8]:
funnel = list(db.repos.aggregate([
    {"$match": {"cohort": {"$in": ["profane", "clean"]}}},
    {"$group": {"_id": {"cohort": "$cohort", "status": "$status"}, "n": {"$sum": 1}}},
]))
funnel_df = pd.DataFrame([
    {"cohort": r["_id"]["cohort"], "status": r["_id"]["status"], "n": int(r["n"])}
    for r in funnel
])
order = ["done", "failed", "missing"]
piv = funnel_df.pivot(index="status", columns="cohort", values="n").reindex(order)

fig, ax = plt.subplots(figsize=(16, 9))
x = np.arange(len(piv.index))
w = 0.4
ax.bar(x - w/2, piv["clean"], w, color=ph.CLEAN_COLOR, label="clean")
ax.bar(x + w/2, piv["profane"], w, color=ph.PROFANE_COLOR, label="profane")
ax.set_xticks(x)
ax.set_xticklabels(piv.index)
ax.set_ylabel("repos")
ax.set_title("Cohort funnel — Stage 4 outcomes per cohort")
for i, status in enumerate(piv.index):
    ax.text(i - w/2, piv.loc[status, "clean"] + 5, str(int(piv.loc[status, "clean"])),
            ha="center", va="bottom", fontsize=11, color=ph.FIIT_DARK)
    ax.text(i + w/2, piv.loc[status, "profane"] + 5, str(int(piv.loc[status, "profane"])),
            ha="center", va="bottom", fontsize=11, color=ph.FIIT_DARK)
ax.legend(loc="upper right")
ph.save_plot(fig, "fig01_cohort_funnel", PLOT_DIR)
print("saved fig01_cohort_funnel.png")

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


saved fig01_cohort_funnel.png


### fig 02 — LOC overlay (log-scale histogram)

Confirms bin-matching held: medians within ~12 %. Long tail is longer
on the profane side; rank tests handle that.

In [9]:
fig, ax = plt.subplots(figsize=(16, 9))
loc = df.dropna(subset=["loc_total"])
loc_clean = loc.loc[loc.cohort == "clean", "loc_total"].astype(float)
loc_prof = loc.loc[loc.cohort == "profane", "loc_total"].astype(float)
bins = np.logspace(2, 7.5, 50)
ax.hist(loc_clean, bins=bins, color=ph.CLEAN_COLOR, alpha=0.55, label=f"clean (n={len(loc_clean)})")
ax.hist(loc_prof, bins=bins, color=ph.PROFANE_COLOR, alpha=0.55, label=f"profane (n={len(loc_prof)})")
ax.set_xscale("log")
ax.set_xlabel("loc_total (log scale)")
ax.set_ylabel("repos")
ax.set_title("LOC distribution per cohort — bin-matching held under the rank test")
ax.axvline(np.median(loc_clean), color=ph.CLEAN_COLOR, linestyle="--", linewidth=1.2)
ax.axvline(np.median(loc_prof), color=ph.PROFANE_COLOR, linestyle="--", linewidth=1.2)
ax.legend(loc="upper right")
ph.save_plot(fig, "fig02_loc_overlay", PLOT_DIR)
print(f"medians: clean={np.median(loc_clean):.0f}  profane={np.median(loc_prof):.0f}")

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


medians: clean=17990  profane=19989


### fig 03 — `lizard_avg_ccn` ECDF (right pane of Slide 50)

Visual analogue of the rank test: the profane curve sits to the right
of the clean curve across the whole range, and the gap is widest in
the middle of the distribution.

In [10]:
fig, ax = plt.subplots(figsize=(9.6, 10.8))
sub = df.dropna(subset=["lizard_avg_ccn"])
sns.ecdfplot(data=sub, x="lizard_avg_ccn", hue="cohort",
             palette={"clean": ph.CLEAN_COLOR, "profane": ph.PROFANE_COLOR},
             linewidth=2.5, ax=ax)
ax.set_xlim(1, 8)
ax.set_xlabel("lizard_avg_ccn")
ax.set_ylabel("ECDF")
ax.set_title("Cyclomatic complexity (mean) — ECDF per cohort")
r = ph.mwu_one(df, "lizard_avg_ccn")
caption = (f"MWU U={r['U']:.0f}  p={r['p']:.2g}  "
           f"r_rb={r['r_rb']:.3f}  α={ALPHA_CORRECTED:.2f}")
ax.text(0.97, 0.05, caption, transform=ax.transAxes, ha="right", va="bottom",
        fontsize=11, color=ph.FIIT_DARK,
        bbox=dict(facecolor="white", edgecolor=ph.FIIT_GRAY, boxstyle="round,pad=0.4"))
ph.save_plot(fig, "fig03_lizard_avg_ecdf", PLOT_DIR)

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


PosixPath('../presentation/public/images/plots/fig03_lizard_avg_ecdf.png')

### fig 04 — `lizard_ccn_p99` ECDF

Tail-end complexity. The strongest of the three significant lizard
results (`p = 6.6 × 10⁻⁵`).

In [11]:
fig, ax = plt.subplots(figsize=(16, 9))
sub = df.dropna(subset=["lizard_ccn_p99"])
sns.ecdfplot(data=sub, x="lizard_ccn_p99", hue="cohort",
             palette={"clean": ph.CLEAN_COLOR, "profane": ph.PROFANE_COLOR},
             linewidth=2.5, ax=ax)
ax.set_xlim(0, 60)
ax.set_xlabel("lizard_ccn_p99 (worst-1 % function complexity)")
ax.set_ylabel("ECDF")
ax.set_title("Tail-end cyclomatic complexity — 99th percentile per repo")
r = ph.mwu_one(df, "lizard_ccn_p99")
caption = (f"MWU U={r['U']:.0f}  p={r['p']:.2g}  r_rb={r['r_rb']:.3f}")
ax.text(0.97, 0.05, caption, transform=ax.transAxes, ha="right", va="bottom",
        fontsize=11, color=ph.FIIT_DARK,
        bbox=dict(facecolor="white", edgecolor=ph.FIIT_GRAY, boxstyle="round,pad=0.4"))
ph.save_plot(fig, "fig04_lizard_p99_ecdf", PLOT_DIR)

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


PosixPath('../presentation/public/images/plots/fig04_lizard_p99_ecdf.png')

### fig 05 — quality forest plot (left pane of Slide 50)

Five a-priori metrics, rank-biserial effect size on x-axis with bootstrap
95 % CI. Significance markers reflect Bonferroni at α/5 = 0.01.

In [12]:
rows = []
for m in A_PRIORI:
    sub = df[["cohort", m]].dropna()
    a = sub.loc[sub.cohort == "clean", m].astype(float).to_numpy()
    b = sub.loc[sub.cohort == "profane", m].astype(float).to_numpy()
    r = ph.mwu_one(df, m)
    lo, hi = ph.bootstrap_ci_rb(a, b, n_resamples=1000, random_state=42)
    rows.append({**r, "ci_lo": lo, "ci_hi": hi})
forest = pd.DataFrame(rows).sort_values("r_rb")

fig, ax = plt.subplots(figsize=(9.6, 10.8))
y = np.arange(len(forest))
sig = (forest["p"] < ALPHA_CORRECTED).to_numpy()
forest_r = forest["r_rb"].to_numpy()
ci_lo = forest["ci_lo"].to_numpy()
ci_hi = forest["ci_hi"].to_numpy()
for i in range(len(forest)):
    color = ph.PROFANE_COLOR if sig[i] else ph.FIIT_GRAY
    ax.errorbar(forest_r[i], y[i],
                xerr=[[forest_r[i] - ci_lo[i]], [ci_hi[i] - forest_r[i]]],
                fmt="o", color=color, ecolor=color,
                elinewidth=2.4, capsize=6, markersize=12, zorder=3)
for yi, p, s in zip(y, forest["p"], sig):
    marker = "✓ sig." if s else "n.s."
    ax.text(0.66, yi, f"p={p:.2g}  {marker}", va="center", fontsize=11,
            color=ph.FIIT_DARK)
ax.axvline(0, color=ph.FIIT_GRAY, linestyle="--", linewidth=1.0)
ax.set_yticks(y)
ax.set_yticklabels(forest["metric"])
ax.set_xlim(-0.25, 0.85)
ax.set_xlabel("rank-biserial r_rb (positive ⇒ profane > clean)")
ax.set_title("Effect sizes across five a-priori metrics — Bonferroni α/5 = 0.01")
ph.save_plot(fig, "fig05_quality_forest", PLOT_DIR)
forest

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


,metric,n_clean,n_prof,median_clean,median_prof,U,p,r_rb,ci_lo,ci_hi
4,comment_to_code_ratio,684,604,0.018597,0.015827,211431.5,0.464886,-0.023544,-0.085609,0.040629
3,jscpd_duplicate_rate,653,567,0.048642,0.047094,184318.0,0.894929,0.004362,-0.062307,0.070125
2,ruff_issues_per_kloc,112,96,38.222499,48.487166,5068.5,0.478053,0.057199,-0.111235,0.223821
0,lizard_avg_ccn,570,508,2.132862,2.333236,125807.0,0.000201,0.131047,0.061640,0.203464
1,lizard_ccn_p99,565,502,10.000000,12.550000,121773.5,0.000066,0.141321,0.075074,0.205579


### fig 06 — top-30 profanity words

Bar chart from `commit_stats.profanity_top` (post-LDNOOBW false-positive
cluster `xxx`/`xx` noted in the deck speaker notes).

In [13]:
top_prof = list(db.repos.aggregate([
    {"$match": {"commit_stats.profanity_hits": {"$gte": 1}}},
    {"$project": {"words": {"$objectToArray": "$commit_stats.profanity_top"}}},
    {"$unwind": "$words"},
    {"$group": {"_id": "$words.k", "n": {"$sum": "$words.v"}}},
    {"$sort": {"n": -1}},
    {"$limit": 30},
]))
prof_df = pd.DataFrame([{"word": r["_id"], "n": int(r["n"])} for r in top_prof])
prof_df = prof_df.iloc[::-1]

fig, ax = plt.subplots(figsize=(16, 9))
ax.barh(prof_df["word"], prof_df["n"], color=ph.PROFANE_COLOR)
ax.set_xlabel("commits")
ax.set_title("Top-30 profanity words across 3.7 M repos (LDNOOBW match)")
for i, (w, n) in enumerate(zip(prof_df["word"], prof_df["n"])):
    ax.text(n + 50, i, f"{n:,}", va="center", fontsize=10, color=ph.FIIT_DARK)
ax.set_xlim(0, prof_df["n"].max() * 1.15)
ph.save_plot(fig, "fig06_top_profanity", PLOT_DIR)

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


PosixPath('../presentation/public/images/plots/fig06_top_profanity.png')

### fig 07 — top-30 emoji

In [14]:
top_emoji = list(db.repos.aggregate([
    {"$match": {"commit_stats.emoji_hits": {"$gte": 1}}},
    {"$project": {"emoji": {"$objectToArray": "$commit_stats.emoji_top"}}},
    {"$unwind": "$emoji"},
    {"$group": {"_id": "$emoji.k", "n": {"$sum": "$emoji.v"}}},
    {"$sort": {"n": -1}},
    {"$limit": 30},
]))
emoji_df = pd.DataFrame([{"emoji": r["_id"], "n": int(r["n"])} for r in top_emoji])
emoji_df = emoji_df.iloc[::-1]

fig, ax = plt.subplots(figsize=(16, 9))
ax.barh(emoji_df["emoji"], emoji_df["n"], color=ph.PROFANE_COLOR)
ax.set_xlabel("commits")
ax.set_title("Top-30 emoji across 3.7 M repos")
for i, (e, n) in enumerate(zip(emoji_df["emoji"], emoji_df["n"])):
    ax.text(n + 100, i, f"{n:,}", va="center", fontsize=10, color=ph.FIIT_DARK)
ax.set_xlim(0, emoji_df["n"].max() * 1.15)
ph.save_plot(fig, "fig07_top_emoji", PLOT_DIR)

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


/Users/jdubec/Projects/OpenSource/oss-profanity/notebooks/_plot_helpers.py:134: UserWarning: Glyph 128214 (\N{OPEN BOOK}) missing from font(s) DejaVu Sans, DejaVu Sans.
  fig.savefig(out)
findfont: Font family 'Open Sans' not found.


/Users/jdubec/Projects/OpenSource/oss-profanity/notebooks/_plot_helpers.py:134: UserWarning: Glyph 10133 (\N{HEAVY PLUS SIGN}) missing from font(s) DejaVu Sans, DejaVu Sans.
  fig.savefig(out)
findfont: Font family 'Open Sans' not found.


/Users/jdubec/Projects/OpenSource/oss-profanity/notebooks/_plot_helpers.py:134: UserWarning: Glyph 128278 (\N{BOOKMARK}) missing from font(s) DejaVu Sans, DejaVu Sans.
  fig.savefig(out)
findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


PosixPath('../presentation/public/images/plots/fig07_top_emoji.png')

### fig 08 — Python subset boxplot (`ruff_issues_per_kloc`)

Direction visible (profane higher in median), not significant under
Bonferroni. Honest negative result, paired with the matching
hypothesis test in cell 5.

In [15]:
fig, ax = plt.subplots(figsize=(16, 9))
py_sub = df[df.primary_language == "python"][["cohort", "ruff_issues_per_kloc"]].dropna()
sns.boxplot(data=py_sub, x="cohort", y="ruff_issues_per_kloc",
            order=["clean", "profane"],
            palette={"clean": ph.CLEAN_COLOR, "profane": ph.PROFANE_COLOR},
            ax=ax, fliersize=3, linewidth=1.2)
ax.set_yscale("symlog", linthresh=10)
ax.set_xlabel("")
ax.set_ylabel("ruff_issues_per_kloc (symlog)")
n_clean = (py_sub.cohort == "clean").sum()
n_prof = (py_sub.cohort == "profane").sum()
ax.set_title(f"Python subset — ruff issues per kLOC  (n_clean={n_clean}, n_prof={n_prof})")
r = ph.mwu_one(df_py, "ruff_issues_per_kloc")
caption = f"MWU p={r['p']:.2g}  r_rb={r['r_rb']:.3f}  not significant under α/5"
ax.text(0.50, -0.15, caption, transform=ax.transAxes, ha="center",
        fontsize=11, color=ph.FIIT_DARK)
ph.save_plot(fig, "fig08_python_subset_box", PLOT_DIR)

/var/folders/pn/gvsh7sg50897b143wlyzqzd80000gn/T/ipykernel_87627/2197946984.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=py_sub, x="cohort", y="ruff_issues_per_kloc",
findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


...

findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


findfont: Font family 'Open Sans' not found.


PosixPath('../presentation/public/images/plots/fig08_python_subset_box.png')

## 9 · Save the numerical record

`presentation/results.json` is the regenerable single-source-of-truth for
every number on a slide. Gitignored; built fresh by re-running this notebook.

In [16]:
payload = {
    "alpha": ALPHA,
    "alpha_corrected": ALPHA_CORRECTED,
    "bonferroni_denominator": len(A_PRIORI),
    "a_priori_metrics": A_PRIORI,
    "n_done": int((df.cohort == "clean").sum() + (df.cohort == "profane").sum()),
    "n_clean": int((df.cohort == "clean").sum()),
    "n_profane": int((df.cohort == "profane").sum()),
    "missingness": summary.to_dict("records"),
    "pooled": results_pooled.to_dict("records"),
    "python_subset": results_py.to_dict("records"),
    "js_subset": results_js.to_dict("records"),
    "descriptive": descriptive.to_dict("records"),
    "forest": forest.to_dict("records"),
}
with RESULTS_JSON.open("w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
print(f"saved {RESULTS_JSON}")

saved ../presentation/results.json


## Headline (one sentence)

> **Profane repos have ~10 % higher median cyclomatic complexity
> (p < 10⁻⁴, small effect r_rb ≈ 0.13). Lint counts, clone rate, and
> comment density show no significant cohort difference.**

The forest plot (fig 05) and the ECDF (fig 03) are the two anchors for
that sentence. ESLint sits as a known data hole; the highest-priority
follow-up is in `docs/IDEAS.md`.